# IAA Calculator
This notebook is used to calculate the IAA between two annotators

In [5]:
from seqeval.metrics import accuracy_score
from seqeval.metrics import classification_report
from seqeval.metrics import f1_score
from seqeval.scheme import IOB2
from nervaluate.evaluator import Evaluator
import pandas as pd
from sklearn.metrics import cohen_kappa_score
# from sklearn.metrics import classification_report

In [ ]:
# pred_sents, pred_labels = get_list_of_sentences(best_final)
# test_set = r"/Users/manyawalavalkar/Desktop/thesis_files/thesis-ner-manya-explore/datasets/named/F.conll"
# test_sents, test_labels = get_list_of_sentences(test_set)
# pred_set = r"/Users/manyawalavalkar/Desktop/thesis_files/thesis-ner-manya-explore/datasets/named/J.conll"
# pred_sents, pred_labels = get_list_of_sentences(pred_set)

labels = ["COURT", "PERSON", "ORG", "GPE", "LAW", "DATE", "TAX_TYPE", "TAX_CONCEPT", "PROVISION", "JURISDICTION"]

from nervaluate import Evaluator

f_file = r"/Users/manyawalavalkar/Desktop/thesis_files/thesis-ner-manya-explore/datasets/named/F.conll"
j_file = r"/Users/manyawalavalkar/Desktop/thesis_files/thesis-ner-manya-explore/datasets/named/J.conll"

with open(f_file, "r", encoding="utf-8") as f:
    annotator_a = f.read()

with open(j_file, "r", encoding="utf-8") as f:
    annotator_b = f.read()

evaluator = Evaluator(annotator_a, annotator_b, tags=labels, loader="conll")
results = evaluator.evaluate()
strict = results["overall"]["strict"]
print("precision:", round(strict.precision, 3))
print("recall:", round(strict.recall, 3))
print("f1:", round(strict.f1, 3))

precision: 0.693
recall: 0.625
f1: 0.657


In [ ]:
labels = ["COURT", "PERSON", "ORG", "GPE", "LAW", "DATE", "TAX_TYPE", "TAX_CONCEPT", "PROVISION", "JURISDICTION"]

def read_conll(path):
    """this reads a conll file and returns sentences as (token, label) pairs."""
    sentences = []
    current_sentence = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.rstrip("\n")
            if line == "":
                if current_sentence:
                    sentences.append(current_sentence)
                    current_sentence = []
            else:
                parts = line.split("\t")
                token = parts[0]
                label = parts[1]
                current_sentence.append((token, label))
    if current_sentence:
        sentences.append(current_sentence)
    return sentences


def get_entities(sentences):
    """this converts bio labels to start, end, label entity spans."""
    entities = []
    for sentence_id, sentence in enumerate(sentences): #for every sentence
        current_label = None #ent being tracked
        start = None #start index of current index

        for token_id, (token, tag) in enumerate(sentence): #get the token id, token, and tag per sentence
            if tag.startswith("B-"): # if the label starts with B- 
                if current_label is not None: # if tracking an entity already
                    entities.append((sentence_id, start, token_id - 1, current_label)) #add its info
                current_label = tag[2:] #set the current label value
                start = token_id #and the token position 
            elif tag.startswith("I-"): #if it starts with I
                if current_label is None: #and if no ent already being tracked
                    current_label = tag[2:] #then this is the current label value
                    start = token_id #and the token position
            else: #else if the label is O
                if current_label is not None: #if an ent is currently being tracked
                    entities.append((sentence_id, start, token_id - 1, current_label)) #close and save the entity
                #reset
                current_label = None 
                start = None
        #if an ent continues to the end, close at final token
        if current_label is not None:
            entities.append((sentence_id, start, len(sentence) - 1, current_label))

    return entities

f_sentences = read_conll(f_file)
j_sentences = read_conll(j_file)

f_entities = get_entities(f_sentences)
j_entities = get_entities(j_sentences)

print("F entities:", len(f_entities))
print("J entities:", len(j_entities))


F entities: 112
J entities: 101


In [20]:
#looking at shared spans between the two annotators and calculating the cohens k
f_span_labels = {}
for sentence_id, start, end, label in f_entities: #get starting and ending positions for entities
    span = (sentence_id, start, end)
    f_span_labels[span] = label #add each span and its label as k, v pairs
j_span_labels = {}

for sentence_id, start, end, label in j_entities: #same here
    span = (sentence_id, start, end)
    j_span_labels[span] = label
shared_spans = set(f_span_labels.keys()) & set(j_span_labels.keys()) #now we can make a set of the shared spans
print("shared spans:", len(shared_spans))

f_labels = []
j_labels = []
for span in shared_spans:
    f_labels.append(f_span_labels[span])
    j_labels.append(j_span_labels[span])
kappa = cohen_kappa_score(f_labels, j_labels) #calculate the cohens kappa of the shared spans
print("cohen's kappa on shared spans", round(kappa, 3))

shared spans: 74
cohen's kappa on shared spans 0.916


In [17]:
#just checking the spans that have different labels
for span in sorted(shared_spans):
    f_label = f_span_labels[span]
    j_label = j_span_labels[span]

    if f_label != j_label:
        print(span, "annotator F:", f_label, "annotator J:", j_label)

(1, 13, 14) annotator F: PROVISION annotator J: LAW
(3, 187, 188) annotator F: JURISDICTION annotator J: ORG
(3, 508, 509) annotator F: PROVISION annotator J: LAW
(4, 80, 81) annotator F: TAX_CONCEPT annotator J: TAX_TYPE
